# Parse PCAP into a pandas DataFrame

This notebook reads a `.pcap` file, extracts useful packet fields, and loads them into a `pandas` DataFrame for analysis.

In [ ]:
# If needed, uncomment and run:
# %pip install scapy pandas

In [1]:
from pathlib import Path
from datetime import datetime

import pandas as pd
from scapy.all import Ether, IP, IPv6, TCP, UDP, ICMP, DNS, rdpcap

In [2]:
# Set your input PCAP path here
pcap_path = Path("recording-marios_foo-merged.pcap")

if not pcap_path.exists():
    raise FileNotFoundError(f"PCAP file not found: {pcap_path.resolve()}")

packets = rdpcap(str(pcap_path))
len(packets)

256

In [3]:
def parse_packet(pkt):
    row = {
        "timestamp": datetime.fromtimestamp(float(pkt.time)),
        "length": len(pkt),
        "eth_src": None,
        "eth_dst": None,
        "ip_version": None,
        "src_ip": None,
        "dst_ip": None,
        "protocol": None,
        "src_port": None,
        "dst_port": None,
        "tcp_flags": None,
        "icmp_type": None,
        "dns_qname": None,
        "l7_raw_payload": None,
    }

    if Ether in pkt:
        row["eth_src"] = pkt[Ether].src
        row["eth_dst"] = pkt[Ether].dst

    if IP in pkt:
        ip = pkt[IP]
        row["ip_version"] = 4
        row["src_ip"] = ip.src
        row["dst_ip"] = ip.dst
        row["protocol"] = ip.proto
    elif IPv6 in pkt:
        ip6 = pkt[IPv6]
        row["ip_version"] = 6
        row["src_ip"] = ip6.src
        row["dst_ip"] = ip6.dst
        row["protocol"] = ip6.nh

    transport_payload = None

    if TCP in pkt:
        tcp = pkt[TCP]
        row["protocol"] = "TCP"
        row["src_port"] = int(tcp.sport)
        row["dst_port"] = int(tcp.dport)
        row["tcp_flags"] = str(tcp.flags)
        transport_payload = bytes(tcp.payload)
    elif UDP in pkt:
        udp = pkt[UDP]
        row["protocol"] = "UDP"
        row["src_port"] = int(udp.sport)
        row["dst_port"] = int(udp.dport)
        transport_payload = bytes(udp.payload)
    elif ICMP in pkt:
        icmp = pkt[ICMP]
        row["protocol"] = "ICMP"
        row["icmp_type"] = int(icmp.type)
        transport_payload = bytes(icmp.payload)

    if DNS in pkt and pkt[DNS].qd is not None:
        qname = pkt[DNS].qd.qname
        if isinstance(qname, bytes):
            qname = qname.decode(errors="ignore")
        row["dns_qname"] = str(qname).rstrip(".")

    if transport_payload:
        row["l7_raw_payload"] = transport_payload.hex()

    return row

In [4]:
records = [parse_packet(pkt) for pkt in packets]
df = pd.DataFrame.from_records(records)

# Optional: normalized protocol labels
proto_num_to_name = {1: "ICMP", 6: "TCP", 17: "UDP"}
df["protocol"] = df["protocol"].map(lambda p: proto_num_to_name.get(p, p))

df.head()

,timestamp,length,eth_src,eth_dst,ip_version,src_ip,dst_ip,protocol,src_port,dst_port,tcp_flags,icmp_type,dns_qname,l7_raw_payload
0,2026-03-04 22:01:30.266794,74,00:00:00:00:00:00,00:00:00:00:00:00,4,127.0.0.1,127.0.0.1,TCP,36768,8000,S,None,None,None
1,2026-03-04 22:01:30.266815,74,00:00:00:00:00:00,00:00:00:00:00:00,4,127.0.0.1,127.0.0.1,TCP,8000,36768,SA,None,None,None
2,2026-03-04 22:01:30.266982,151,00:00:00:00:00:00,00:00:00:00:00:00,4,127.0.0.1,127.0.0.1,TCP,36768,8000,PA,None,None,474554202f6865616c74687a20485454502f312e310d0a...
3,2026-03-04 22:01:30.266995,66,00:00:00:00:00:00,00:00:00:00:00:00,4,127.0.0.1,127.0.0.1,TCP,8000,36768,A,None,None,None
4,2026-03-04 22:01:30.266833,66,00:00:00:00:00:00,00:00:00:00:00:00,4,127.0.0.1,127.0.0.1,TCP,36768,8000,A,None,None,None


In [7]:
# Example: find packets targeting a specific HTTP Host header using hex search
hostname = "api.anthropic.com"

# HTTP header format is: Host: <hostname>\r\n
host_header_hex = hostname.encode("utf-8").hex()
host_header_hex_lower = host_header_hex.lower()

host_matches = df[
    df["l7_raw_payload"].fillna("").str.lower().str.contains(host_header_hex_lower, regex=False)
]

display(host_matches[["timestamp", "src_ip", "src_port", "dst_ip", "dst_port", "protocol"]].head(20))
len(host_matches)

,timestamp,src_ip,src_port,dst_ip,dst_port,protocol
75,2026-03-04 22:01:35.893787,10.0.1.46,54738,10.96.107.24,8080,TCP
124,2026-03-04 22:01:51.144755,10.0.1.46,55966,10.96.107.24,8080,TCP
163,2026-03-04 22:02:17.371348,10.0.1.46,42318,10.96.107.24,8080,TCP
216,2026-03-04 22:02:54.643997,10.0.1.46,40916,10.96.107.24,8080,TCP
244,2026-03-04 22:03:14.113142,10.0.1.46,56654,10.96.107.24,8080,TCP


5

In [9]:
# Example: find all packets from a specific source IP
source_ip = "10.96.107.24"

src_matches = df[df["src_ip"] == source_ip]
display(src_matches[["timestamp", "src_ip", "src_port", "dst_ip", "dst_port", "protocol", "length"]].head(50))
len(src_matches)

,timestamp,src_ip,src_port,dst_ip,dst_port,protocol,length
55,2026-03-04 22:01:35.882244,10.96.107.24,8080,10.0.1.46,54738,TCP,74
61,2026-03-04 22:01:35.882508,10.96.107.24,8080,10.0.1.46,54738,TCP,66
68,2026-03-04 22:01:35.884480,10.96.107.24,8080,10.0.1.46,54738,TCP,66
69,2026-03-04 22:01:35.883751,10.96.107.24,8080,10.0.1.46,54738,TCP,254
70,2026-03-04 22:01:35.883858,10.96.107.24,8080,10.0.1.46,54738,TCP,1755
71,2026-03-04 22:01:35.884001,10.96.107.24,8080,10.0.1.46,54738,TCP,66
73,2026-03-04 22:01:35.893787,10.96.107.24,8080,10.0.1.46,54738,TCP,60
76,2026-03-04 22:01:35.893787,10.96.107.24,8080,10.0.1.46,54738,TCP,60
102,2026-03-04 22:01:51.136663,10.96.107.24,8080,10.0.1.46,55966,TCP,74
114,2026-03-04 22:01:51.138415,10.96.107.24,8080,10.0.1.46,55966,TCP,66


41